In [48]:
import pandas as pd
import numpy as np

crime = pd.read_csv('crime.csv')
requests = pd.read_csv('Requests.csv')
lights = pd.read_csv('Lights.csv')
street_lights = pd.read_csv('Street_Lights.csv')

In [49]:
import pandas as pd
import numpy as np
import time

# ---------------------------------------------------------
# 1. 초고속 하버사인(Haversine) 거리 계산 함수 (원형 자르기용)
# ---------------------------------------------------------
def haversine_np(lon1, lat1, lon2, lat2):
    """
    두 위경도 좌표 사이의 거리를 킬로미터(km) 단위로 반환합니다.
    Numpy 벡터화를 사용하여 수십만 건의 행도 0.1초 만에 계산합니다.
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c # 지구 반지름
    return km

# 2. 대용량 CSV 전처리 파이프라인 함수 (사각형 -> 원형)
# ---------------------------------------------------------
def process_7km_radius(file_name, lat_col, lon_col, output_name):
    print(f"📦 [{file_name}] 데이터 로드 및 7km 필터링 시작...")
    start_time = time.time()
    
    # 1. 파일 불러오기 및 위경도 없는 결측치 제거
    df = pd.read_csv(file_name)
    initial_len = len(df)
    df = df.dropna(subset=[lat_col, lon_col])
    
    # USC 중심 좌표
    usc_lat, usc_lon = 34.0224, -118.2851
    
    # [1단계] 사각형(Bounding Box) 자르기
    # 7km는 위도 약 0.063도, 경도 약 0.077도에 해당합니다. 여유를 두고 0.065, 0.08로 자릅니다.
    df_box = df[
        (df[lat_col].between(usc_lat - 0.065, usc_lat + 0.065)) & 
        (df[lon_col].between(usc_lon - 0.08, usc_lon + 0.08))
    ].copy()
    
    # [2단계] 원형(Circle) 자르기
    # 사각형 안에서 살아남은 데이터만 정확한 거리를 계산하여 반경 7km 이내만 남깁니다.
    df_box['distance_to_usc'] = haversine_np(usc_lon, usc_lat, df_box[lon_col], df_box[lat_col])
    df_final = df_box[df_box['distance_to_usc'] <= 7.0].copy()
    
    # 3. 결과 저장 (용량이 훨씬 가벼워진 새로운 파일로 생성)
    df_final.to_csv(output_name, index=False)
    
    end_time = time.time()
    print(f"✅ 완료! {initial_len:,}개 -> {len(df_final):,}개로 압축 (소요시간: {end_time - start_time:.2f}초)")
    print(f"💾 저장된 파일명: {output_name}\n")


# ---------------------------------------------------------
# 3. 실행부 (3개의 대용량 파일에 적용)
# ---------------------------------------------------------
# 각 파일마다 위도, 경도를 나타내는 컬럼명이 다르므로 정확하게 매칭해줍니다.

# 1. 범죄 데이터 (LAT, LON)
process_7km_radius('crime.csv', 'LAT', 'LON', 'crime_7km.csv')

# 2. 311 환경 민원 데이터 (Latitude, Longitude)
process_7km_radius('Requests.csv', 'Latitude', 'Longitude', 'Requests_7km.csv')

# 3. 실시간 고장난 가로등 데이터 (Latitude, Longitude)
process_7km_radius('Lights.csv', 'Latitude', 'Longitude', 'Lights_7km.csv')

📦 [crime.csv] 데이터 로드 및 7km 필터링 시작...
✅ 완료! 1,004,894개 -> 348,967개로 압축 (소요시간: 3.47초)
💾 저장된 파일명: crime_7km.csv

📦 [Requests.csv] 데이터 로드 및 7km 필터링 시작...
✅ 완료! 1,442,347개 -> 397,893개로 압축 (소요시간: 7.12초)
💾 저장된 파일명: Requests_7km.csv

📦 [Lights.csv] 데이터 로드 및 7km 필터링 시작...
✅ 완료! 241,712개 -> 79,742개로 압축 (소요시간: 1.40초)
💾 저장된 파일명: Lights_7km.csv



In [50]:
!pip install pyproj

In [51]:
import pandas as pd
import numpy as np
import time
from pyproj import Transformer

# ---------------------------------------------------------
# [Street_Lights.csv 전용] 좌표 변환 및 7km 필터링 로직
# ---------------------------------------------------------
print("📦 [Street_Lights.csv] 좌표 변환 및 7km 필터링 시작...")
start_time = time.time()

# 1. 파일 불러오기 및 X, Y 결측치 제거
# (데이터에 맞게 X, Y 컬럼명이 다르다면 수정해 주세요)
df_street = pd.read_csv('Street_Lights.csv')
initial_len = len(df_street)
df_street = df_street.dropna(subset=['X', 'Y'])

# 2. X, Y 좌표를 위경도(LAT, LON)로 변환
# EPSG:3857(웹 메르카토르) -> EPSG:4326(WGS84 위경도) 변환기 생성
# 만약 결과 위경도가 LA(34, -118) 근처가 안 나온다면, 
# LA 지역 투영 좌표계인 "EPSG:2229" 또는 "EPSG:26911"로 원본 좌표계를 변경해야 할 수 있습니다.
transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

# 변환된 값을 새로운 컬럼으로 추가
print("🔄 좌표 변환 중... (데이터 크기에 따라 수 초 소요될 수 있습니다)")
df_street['LON'], df_street['LAT'] = transformer.transform(df_street['X'].values, df_street['Y'].values)

# 3. 7km 반경 필터링 (사각형 -> 원형)
usc_lat, usc_lon = 34.0224, -118.2851

# [1단계] 사각형 자르기
df_box = df_street[
    (df_street['LAT'].between(usc_lat - 0.065, usc_lat + 0.065)) & 
    (df_street['LON'].between(usc_lon - 0.08, usc_lon + 0.08))
].copy()

# [2단계] 원형 자르기 (앞서 정의한 haversine_np 함수 사용)
df_box['distance_to_usc'] = haversine_np(usc_lon, usc_lat, df_box['LON'], df_box['LAT'])
df_final = df_box[df_box['distance_to_usc'] <= 10.0].copy()

# 4. 결과 저장
output_name = 'Street_Lights_7km.csv'
df_final.to_csv(output_name, index=False)

end_time = time.time()
print(f"✅ 완료! {initial_len:,}개 -> {len(df_final):,}개로 압축 및 위경도 변환 성공! (소요시간: {end_time - start_time:.2f}초)")
print(f"💾 저장된 파일명: {output_name}")

# 변환된 데이터 샘플 확인
print("\n🔍 변환된 데이터 샘플 (위도, 경도 확인):")
print(df_final[['X', 'Y', 'LAT', 'LON', 'distance_to_usc']].head())

📦 [Street_Lights.csv] 좌표 변환 및 7km 필터링 시작...
🔄 좌표 변환 중... (데이터 크기에 따라 수 초 소요될 수 있습니다)
✅ 완료! 222,007개 -> 66,927개로 압축 및 위경도 변환 성공! (소요시간: 0.58초)
💾 저장된 파일명: Street_Lights_7km.csv

🔍 변환된 데이터 샘플 (위도, 경도 확인):
                  X             Y        LAT         LON  distance_to_usc
11403 -1.317587e+07  4.026347e+06  33.981718 -118.360893         8.318166
11404 -1.317590e+07  4.026351e+06  33.981745 -118.361157         8.336931
11405 -1.317398e+07  4.026358e+06  33.981798 -118.343909         7.050456
11406 -1.317387e+07  4.026359e+06  33.981805 -118.342895         6.978467
11407 -1.317374e+07  4.026360e+06  33.981811 -118.341713         6.895358


In [44]:
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 12.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.4 MB/s eta 0:00:00


In [52]:
import geopandas as gpd
from shapely.geometry import Point

print("🗺️ [export.geojson] 데이터 로드 및 7km 원형 컷팅 시작...")

# 1. GeoJSON 마스터 파일 불러오기
gdf_infra = gpd.read_file('export.geojson')
initial_len = len(gdf_infra)

# 2. USC 중심점 셋업 (위경도: EPSG:4326)
usc_point = gpd.GeoSeries([Point(-118.2851, 34.0224)], crs="EPSG:4326")

# 3. [핵심] 미터(m) 단위 거리 계산을 위해 LA 투영 좌표계(UTM Zone 11N)로 일시적 변환
gdf_infra_proj = gdf_infra.to_crs(epsg=26911)
usc_point_proj = usc_point.to_crs(epsg=26911)

# 4. 중심점(USC)에서 직선거리가 7,000m(7km) 이하인 것만 남기기 (원형 깎기)
gdf_7km_proj = gdf_infra_proj[gdf_infra_proj.geometry.distance(usc_point_proj.iloc[0]) <= 7000]

# 5. 거리 계산이 끝났으니 다시 원래 위경도 좌표계로 원상복구
gdf_7km = gdf_7km_proj.to_crs(epsg=4326)

print(f"✅ 7km 컷팅 완료! 원본 {initial_len}개 -> {len(gdf_7km)}개로 압축됨\n")

# ---------------------------------------------------------
# 6. 잘려진 7km 데이터 안에서 안전(가점) / 위험(감점) 데이터 쪼개기
# ---------------------------------------------------------
print("🔍 안전 거점과 위험 인프라 분류 중...")

# [가점] 안전 거점 분리
safe_conditions = (
    gdf_7km['shop'].isin(['convenience']) |
    gdf_7km['amenity'].isin(['pharmacy', 'police', 'fire_station', 'hospital', 'fuel']) |
    gdf_7km['tourism'].isin(['hotel']) |
    (gdf_7km['opening_hours'] == '24/7')
)
gdf_safe = gdf_7km[safe_conditions].copy()

# [감점] 위험 인프라 분리
risk_conditions = (
    (gdf_7km['abandoned'] == 'yes') |
    (gdf_7km['landuse'] == 'brownfield') |
    (gdf_7km['tunnel'] == 'yes')
)
gdf_risk = gdf_7km[risk_conditions].copy()

print(f"🟢 7km 반경 내 최종 안전 거점(가점) 개수: {len(gdf_safe)}개")
print(f"🔴 7km 반경 내 최종 위험 인프라(감점) 개수: {len(gdf_risk)}개")

🗺️ [export.geojson] 데이터 로드 및 7km 원형 컷팅 시작...
✅ 7km 컷팅 완료! 원본 1101개 -> 696개로 압축됨

🔍 안전 거점과 위험 인프라 분류 중...
🟢 7km 반경 내 최종 안전 거점(가점) 개수: 276개
🔴 7km 반경 내 최종 위험 인프라(감점) 개수: 420개


In [53]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore') # 성가신 경고 메시지 숨기기

print("🧹 데이터 결측치(NaN) 클린업 시작...\n")

# 1. 정리할 7km CSV 파일 목록
csv_files = [
    'crime_7km.csv', 
    'Requests_7km.csv', 
    'Lights_7km.csv', 
    'Street_Lights_7km.csv'
]

for file in csv_files:
    # 데이터 불러오기
    df = pd.read_csv(file)
    initial_null_count = df.isnull().sum().sum()
    
    if initial_null_count > 0:
        # 결측치가 있다면 타입별로 안전하게 채우기
        for col in df.columns:
            if df[col].dtype == 'object':
                # 텍스트(문자열) 데이터의 빈칸은 'Unknown'으로 채움
                df[col] = df[col].fillna('Unknown')
            else:
                # 숫자형 데이터의 빈칸은 0으로 채움 
                # (위경도 NaN은 이미 앞서 제거했으므로 안심해도 됨)
                df[col] = df[col].fillna(0)
        
        # 덮어쓰기 저장
        df.to_csv(file, index=False)
        print(f"✅ [{file}] 클린업 완료: 총 {initial_null_count:,}개의 빵꾸(NaN)를 메웠습니다.")
    else:
        print(f"✨ [{file}] 결측치 없음: 이미 완벽하게 깨끗합니다!")

# ---------------------------------------------------------
# 2. GeoJSON 데이터 (안전/위험 인프라) 결측치 처리
# ---------------------------------------------------------
# 앞서 메모리에 있는 gdf_safe, gdf_risk 데이터프레임을 대상으로 합니다.
# (이전 셀에서 gdf_safe, gdf_risk가 생성되어 있다고 가정)

# OSM 데이터 특성상 'shop'이 있는 곳은 'amenity'가 NaN인 식으로 빈칸이 아주 많습니다.
for col in gdf_safe.columns:
    if col != 'geometry': # 지도 도형 데이터는 건드리지 않음
        gdf_safe[col] = gdf_safe[col].fillna('none')

for col in gdf_risk.columns:
    if col != 'geometry':
        gdf_risk[col] = gdf_risk[col].fillna('none')

print("\n✅ [GeoJSON 인프라 데이터] 클린업 완료: 속성 빈칸을 'none'으로 채웠습니다.")
print("\n🎉 모든 데이터의 Null/NaN 처리 완료! 공간 조인 준비가 끝났습니다.")

🧹 데이터 결측치(NaN) 클린업 시작...

✅ [crime_7km.csv] 클린업 완료: 총 1,889,821개의 빵꾸(NaN)를 메웠습니다.
✅ [Requests_7km.csv] 클린업 완료: 총 787,685개의 빵꾸(NaN)를 메웠습니다.
✅ [Lights_7km.csv] 클린업 완료: 총 340,195개의 빵꾸(NaN)를 메웠습니다.
✨ [Street_Lights_7km.csv] 결측치 없음: 이미 완벽하게 깨끗합니다!

✅ [GeoJSON 인프라 데이터] 클린업 완료: 속성 빈칸을 'none'으로 채웠습니다.

🎉 모든 데이터의 Null/NaN 처리 완료! 공간 조인 준비가 끝났습니다.


In [54]:
import pandas as pd

# 검사할 파일 목록
csv_files = [
    'crime_7km.csv', 
    'Requests_7km.csv', 
    'Lights_7km.csv', 
    'Street_Lights_7km.csv'
]

print("🔍 각 파일별 결측치(NaN) 정밀 검사를 시작합니다...\n")
print("=" * 50)

for file in csv_files:
    try:
        # 파일 불러오기
        df = pd.read_csv(file)
        
        # 컬럼별 결측치 개수 계산
        null_counts = df.isnull().sum()
        total_nulls = null_counts.sum()
        
        if total_nulls == 0:
            print(f"✅ [{file}] 총 결측치: 0개\n   결과: 완벽하게 깨끗합니다!")
        else:
            print(f"⚠️ [{file}] 총 결측치: {total_nulls:,}개 발견!")
            print("   [상세 내역]")
            
            # 결측치가 1개 이상 있는 컬럼만 뽑아서 출력
            missing_cols = null_counts[null_counts > 0]
            for col, count in missing_cols.items():
                print(f"   - '{col}' 컬럼: {count:,}개 빈칸")
                
        print("=" * 50)
        
    except FileNotFoundError:
        print(f"❌ [{file}] 파일을 찾을 수 없습니다. 폴더에 파일이 있는지 확인해 주세요.")
        print("=" * 50)

🔍 각 파일별 결측치(NaN) 정밀 검사를 시작합니다...

✅ [crime_7km.csv] 총 결측치: 0개
   결과: 완벽하게 깨끗합니다!
✅ [Requests_7km.csv] 총 결측치: 0개
   결과: 완벽하게 깨끗합니다!
✅ [Lights_7km.csv] 총 결측치: 0개
   결과: 완벽하게 깨끗합니다!
✅ [Street_Lights_7km.csv] 총 결측치: 0개
   결과: 완벽하게 깨끗합니다!


In [72]:
# ... existing code ...
import pandas as pd
import geopandas as gpd # [Task 4-1] 좌표 변환을 위해 추가

print("🚀 [Step 1] 4대 데이터 피처 엔지니어링 시작 (불필요 컬럼 제거 및 점수화)\n")

# ==============================================================================
# 1. 범죄 데이터 (Crime)
# ==============================================================================
df_crime = pd.read_csv('crime_10km.csv')

# 살릴 컬럼만 선택
# [Task 4-4] Crm Cd (숫자 코드) 추가
crime_cols = ['DATE OCC', 'TIME OCC', 'AREA NAME', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Premis Desc', 'LAT', 'LON']
df_crime = df_crime[crime_cols].copy()

# [Task 4-2] 지오코딩 실패(LAT=0, LON=0) 레코드 명시적 제거
df_crime = df_crime[(df_crime['LAT'] != 0) & (df_crime['LON'] != 0)]

# 야외 범죄만 필터링
outdoor_keywords = 'STREET|ALLEY|SIDEWALK|PARKING|PARK|LOT'
df_crime = df_crime[df_crime['Premis Desc'].str.contains(outdoor_keywords, case=False, na=False)]

# 가중치 텍스트 -> 숫자 변환
df_crime['is_night'] = ((df_crime['TIME OCC'] >= 2000) | (df_crime['TIME OCC'] <= 500)).astype(int)
df_crime['is_severe'] = df_crime['Part 1-2'].astype(str).str.contains('1').astype(int) # 강력범죄

# [Task 4-4] 텍스트 매칭 대신 LAPD 공식 Crm Cd(숫자)로 Fatal(살인, 강간, 강도, 가중폭행) 지정
fatal_codes = [110, 113, 121, 122, 815, 820, 821, 210, 220, 230, 231, 235, 236, 250, 251]
df_crime['is_fatal'] = df_crime['Crm Cd'].isin(fatal_codes).astype(int)

# [최종 범죄 위험도 점수: 1 ~ 6점]
# ... existing code ...
df_crime['crime_score'] = 1 + df_crime['is_night'] + (df_crime['is_severe'] * 2) + (df_crime['is_fatal'] * 2)
df_crime.to_csv('crime_ready.csv', index=False)
print("✅ 1/4 범죄 데이터 (crime_ready.csv) 가공 완료")


# ==============================================================================
# 2. 311 환경 민원 데이터 (Requests)
# ==============================================================================
df_req = pd.read_csv('Requests_10km.csv')

# [Task 1] ML 시계열 예측을 위해 과거 상태 복원용 시간 컬럼들 모두 살리기
req_cols = ['CreatedDate', 'ClosedDate', 'UpdatedDate', 'ServiceDate', 'RequestType', 'Status', 'Latitude', 'Longitude']
available_req_cols = [col for col in req_cols if col in df_req.columns]
df_req = df_req[available_req_cols].copy()

# ⚠️ [Task 1] "현재 방치된(Open/Pending) 민원만 남기기" 삭제 (모든 이력을 살립니다!)

# [Task 3] 노숙자(Homeless/Encampment) 관련 민원은 윤리적/해커톤 방어를 위해 완전 배제!
hazard_keywords = 'Bulky|Dumping|Graffiti|Damage|Pothole'
df_req['is_hazard'] = df_req['RequestType'].str.contains(hazard_keywords, case=False, na=False).astype(int)

# [최종 민원 위험도 점수: 1 ~ 2점]
# ... existing code ...
df_req['request_score'] = 1 + df_req['is_hazard']
df_req.to_csv('requests_ready.csv', index=False)
print("✅ 2/4 민원 데이터 (requests_ready.csv) 가공 완료")


# ==============================================================================
# 3. 고장난 가로등 데이터 (Broken Lights)
# ==============================================================================
df_broken = pd.read_csv('Lights_10km.csv')

# [Task 1] 과거 고장 이력을 위해 ClosedDate, UpdatedDate 살림
broken_cols = ['CreatedDate', 'ClosedDate', 'UpdatedDate', 'RequestType', 'Status', 'Latitude', 'Longitude']
available_broken_cols = [col for col in broken_cols if col in df_broken.columns]
df_broken = df_broken[available_broken_cols].copy()

# ⚠️ [Task 1] "수리 전(Open/Pending)인 위험 구역만 남기기" 삭제 (모든 이력을 살립니다!)

# 고장난 가로등은 무조건 위험 점수 부여
# ... existing code ...
df_broken['broken_light_score'] = 3
df_broken.to_csv('broken_lights_ready.csv', index=False)
print("✅ 3/4 고장 가로등 데이터 (broken_lights_ready.csv) 가공 완료")


# ==============================================================================
# 4. 전체 가로등 인프라 현황 (Street Lights)
# ==============================================================================
df_lights = pd.read_csv('Street_lights_7km.csv')

# [Task 4-1] 좌표계 EPSG:2229(Feet) 여부 확인 및 4326(Lat/Lon) 자동 변환
# LA 오픈데이터의 X 좌표가 -118.x가 아니라 수백만 단위(Feet)라면 변환을 수행합니다.
if 'X' in df_lights.columns and 'Y' in df_lights.columns:
    if df_lights['X'].max() > 180 or df_lights['X'].min() < -180:
        print("  🔄 가로등 좌표 EPSG:2229 감지 -> EPSG:4326으로 변환합니다...")
        gdf_lights = gpd.GeoDataFrame(df_lights, geometry=gpd.points_from_xy(df_lights['X'], df_lights['Y']), crs="EPSG:2229")
        gdf_lights = gdf_lights.to_crs("EPSG:4326")
        df_lights['Longitude'] = gdf_lights.geometry.x
        df_lights['Latitude'] = gdf_lights.geometry.y
    else:
        df_lights.rename(columns={'Y': 'Latitude', 'X': 'Longitude'}, inplace=True)
elif 'LAT' in df_lights.columns and 'LON' in df_lights.columns:
    df_lights.rename(columns={'LAT': 'Latitude', 'LON': 'Longitude'}, inplace=True)

# 살릴 컬럼만 선택 (공간 조인에 필요한 위경도만 우선 추출)
light_cols = ['Latitude', 'Longitude']
available_light_cols = [col for col in light_cols if col in df_lights.columns]
df_lights = df_lights[available_light_cols].copy()

# 결측치(좌표 없는 가로등) 제거
df_lights = df_lights.dropna(subset=['Latitude', 'Longitude'])

# [방어 점수 부여] 정상 작동하는 일반 가로등은 위험도를 낮추는 방어 점수(-1) 부여
df_lights['defense_score'] = -1

# 최종 저장
df_lights.to_csv('street_lights_ready.csv', index=False)
print("✅ 4/4 가로등 인프라 데이터 (street_lights_ready.csv) 가공 완료\n")

print("🎉 모든 피처 엔지니어링이 성공적으로 완료되었습니다!")

🚀 [Step 1] 4대 데이터 피처 엔지니어링 시작 (불필요 컬럼 제거 및 점수화)

✅ 1/4 범죄 데이터 (crime_ready.csv) 가공 완료
✅ 2/4 민원 데이터 (requests_ready.csv) 가공 완료
✅ 3/4 고장 가로등 데이터 (broken_lights_ready.csv) 가공 완료
  🔄 가로등 좌표 EPSG:2229 감지 -> EPSG:4326으로 변환합니다...
✅ 4/4 가로등 인프라 데이터 (street_lights_ready.csv) 가공 완료

🎉 모든 피처 엔지니어링이 성공적으로 완료되었습니다!


도로망 추출 코드

In [60]:
!pip install geopandas
!pip install osmnx

In [61]:
import osmnx as ox
import geopandas as gpd
import time

print("🌐 [Step 2] USC 반경 7km 보행자 도로망(Graph) 추출 시작...")
start_time = time.time()

# 1. USC 중심 좌표 및 반경 설정
# 앞서 데이터 용량을 줄이기 위해 7km로 통일했으므로 7000미터로 설정합니다.
usc_lat, usc_lon = 34.0224, -118.2851
radius_meters = 7000

try:
    # 2. OSM 서버에서 보행자 도로망 다운로드 (핵심)
    # network_type='walk' : 자동차 도로 제외, 오직 사람이 걸을 수 있는 인도, 횡단보도, 골목 등만 추출
    print("⏳ 오픈스트리트맵(OSM) 서버에서 데이터를 다운로드 중입니다. (1~3분 소요)")
    G = ox.graph_from_point((usc_lat, usc_lon), dist=radius_meters, network_type='walk')
    
    # 3. 다운받은 네트워크(Graph)를 점(nodes)과 선(edges) 데이터프레임으로 분리
    # 우리는 길(선) 위에 점수를 매길 것이므로 edges가 핵심입니다.
    nodes, edges = ox.graph_to_gdfs(G)
    
    # 4. 저장(GeoJSON)을 위한 전처리 (에러 방지용)
    # OSMnx 데이터 중 일부(예: highway 종류)가 리스트[list] 형태로 들어있어서 
    # 그냥 저장하면 에러가 납니다. 전부 문자열(string)로 변환해 줍니다.
    for col in edges.columns:
        if any(isinstance(val, list) for val in edges[col]):
            edges[col] = edges[col].astype(str)
            
    # 5. 파일로 저장 (다음 단계인 공간 조인을 위해 보관)
    # GeoJSON 형식은 지도 시각화와 공간 데이터 처리에 가장 널리 쓰이는 포맷입니다.
    output_filename = "usc_walkways_7km.geojson"
    edges.to_file(output_filename, driver='GeoJSON')
    
    end_time = time.time()
    print(f"\n🎉 도로망 추출 및 저장 성공!")
    print(f"✅ 총 {len(edges):,}개의 걸을 수 있는 도로 구간(Edge)이 확보되었습니다.")
    print(f"💾 파일명: {output_filename} (소요시간: {end_time - start_time:.2f}초)")

except Exception as e:
    print(f"\n⚠️ [에러 발생] 인터넷 연결이나 OSMnx 라이브러리를 확인해 주세요.")
    print(f"에러 내용: {e}")

🌐 [Step 2] USC 반경 7km 보행자 도로망(Graph) 추출 시작...
⏳ 오픈스트리트맵(OSM) 서버에서 데이터를 다운로드 중입니다. (1~3분 소요)

🎉 도로망 추출 및 저장 성공!
✅ 총 195,320개의 걸을 수 있는 도로 구간(Edge)이 확보되었습니다.
💾 파일명: usc_walkways_7km.geojson (소요시간: 33.15초)


공간 조인

In [69]:
import pandas as pd
import geopandas as gpd
import numpy as np
import time

print("🚀 [Step 3] 공간 조인(Spatial Join) 및 가중치 정규화 시작\n")
start_time = time.time()

CRS_M = "EPSG:26911"
BUFFER_M = 50
DENSITY_MIN_LEN = 50.0   # 밀도 분모 하한 = 버퍼 반경 (10m edge 밀도 폭발 방지)

# ─────────────────────────────────────────────
# 1. 도로망 로드 및 정제
# ─────────────────────────────────────────────
print("1. 도로망 데이터 로드 및 정제 중...")
gdf_edges = gpd.read_file("usc_walkways_7km.geojson")

assert {'u', 'v'}.issubset(gdf_edges.columns), "u, v 컬럼 없음!"
if 'key' not in gdf_edges.columns:
    gdf_edges['key'] = 0

if 'length' not in gdf_edges.columns:
    gdf_edges['length'] = gdf_edges.to_crs(CRS_M).geometry.length

gdf_edges['undir_id'] = gdf_edges.apply(
    lambda r: f"{tuple(sorted([str(r['u']), str(r['v'])]))}_{r['key']}", axis=1
)

gdf_unique_edges = gdf_edges.drop_duplicates(subset=['undir_id']).copy()
gdf_unique_edges = gdf_unique_edges.reset_index(drop=True)
gdf_unique_edges['edge_id'] = gdf_unique_edges.index
print(f"   -> 양방향 dedup: {len(gdf_edges):,}개 → {len(gdf_unique_edges):,}개")

gdf_edges_m = gdf_unique_edges.to_crs(CRS_M).copy()
gdf_edges_m['geometry'] = gdf_edges_m.geometry.buffer(BUFFER_M)
print(f"   -> {CRS_M} 변환 및 {BUFFER_M}m 버퍼 생성 완료\n")

# ─────────────────────────────────────────────
# 2. 좌표 자동 진단 + 공간 조인 헬퍼
# ─────────────────────────────────────────────
def fix_coords(df, lat_col, lon_col, csv_file):
    """LA 지역 좌표 범위(위도 33~35, 경도 -119~-117) 검증 및 자동 교정"""
    lat, lon = df[lat_col], df[lon_col]
    lat_ok = lat.between(33, 35).mean() > 0.9
    lon_ok = lon.between(-119, -117).mean() > 0.9

    if lat_ok and lon_ok:
        return df, "정상"

    # 케이스 1: 위/경도 스왑 (lat 자리에 -118.xx, lon 자리에 34.xx)
    if lat.between(-119, -117).mean() > 0.9 and lon.between(33, 35).mean() > 0.9:
        df = df.rename(columns={lat_col: lon_col, lon_col: lat_col})
        return df, "⚠️ 위/경도 스왑 감지 → 자동 교정"

    # 케이스 2: 투영좌표(Web Mercator) 그대로 저장됨 (값이 수백만 단위)
    if lat.abs().max() > 180 or lon.abs().max() > 180:
        from pyproj import Transformer
        tf = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
        lon_new, lat_new = tf.transform(df[lon_col].values, df[lat_col].values)
        df[lon_col], df[lat_col] = lon_new, lat_new
        # 변환 후 재검증
        if df[lat_col].between(33, 35).mean() > 0.9:
            return df, "⚠️ Web Mercator 좌표 감지 → EPSG:4326 자동 변환"
        return df, "❌ 3857 변환 실패 — X/Y 컬럼 순서 확인 필요"

    return df, f"❌ 좌표 이상 — lat범위({lat.min():.2f}~{lat.max():.2f}), lon범위({lon.min():.2f}~{lon.max():.2f})"


def join_points_to_edges(csv_file, lat_col, lon_col, score_col, result_col_name):
    try:
        df = pd.read_csv(csv_file)
        if df.empty:
            return pd.Series(0, index=gdf_unique_edges['edge_id'], name=result_col_name)

        df, coord_status = fix_coords(df, lat_col, lon_col, csv_file)
        df = df.dropna(subset=[lat_col, lon_col])

        gdf_pts = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
            crs="EPSG:4326"
        ).to_crs(CRS_M)

        joined = gpd.sjoin(gdf_pts, gdf_edges_m[['edge_id', 'geometry']],
                           how='inner', predicate='within')

        scores = joined.groupby('edge_id')[score_col].sum()
        scores.name = result_col_name
        n_mapped = joined['edge_id'].nunique()
        print(f"   ✓ {csv_file}: 좌표[{coord_status}] | 점 {len(df):,}개 → 도로 {n_mapped:,}개 매핑")
        if n_mapped == 0:
            print(f"     ❌ 0개 매핑! {lat_col}/{lon_col} 값 샘플: {df[[lat_col, lon_col]].head(3).values.tolist()}")
        return scores

    except Exception as e:
        print(f"   ⚠️ {csv_file} 에러: {e}")
        return pd.Series(0, index=gdf_unique_edges['edge_id'], name=result_col_name)

print("2. 4가지 점 데이터를 도로망에 조인 중...")
crime_scores   = join_points_to_edges('crime_ready.csv',         'LAT',      'LON',       'crime_score',        'total_crime_score')
request_scores = join_points_to_edges('requests_ready.csv',      'Latitude', 'Longitude', 'request_score',      'total_request_score')
broken_scores  = join_points_to_edges('broken_lights_ready.csv', 'Latitude', 'Longitude', 'broken_light_score', 'total_broken_score')
defense_scores = join_points_to_edges('street_lights_ready.csv', 'Latitude', 'Longitude', 'defense_score',      'total_defense_score')

# ─────────────────────────────────────────────
# 3. 점수 매핑 + 밀도 정규화
# ─────────────────────────────────────────────
print("\n3. 점수를 양방향 도로망에 매핑 및 밀도 정규화...")
gdf_unique_edges = gdf_unique_edges.set_index('edge_id')
gdf_unique_edges = gdf_unique_edges.join(
    [crime_scores, request_scores, broken_scores, defense_scores]
).fillna(0)

score_cols = ['total_crime_score', 'total_request_score',
              'total_broken_score', 'total_defense_score']
gdf_edges = gdf_edges.merge(
    gdf_unique_edges[['undir_id'] + score_cols], on='undir_id', how='left'
)
gdf_edges[score_cols] = gdf_edges[score_cols].fillna(0)

gdf_edges['raw_danger_score'] = gdf_edges[score_cols].sum(axis=1)

# 밀도 분모 하한 = 버퍼 반경(50m): 10m짜리 edge도 실제론 50m 반경 점수를 수집하므로
effective_len = gdf_edges['length'].clip(lower=DENSITY_MIN_LEN)
gdf_edges['final_danger_score'] = (gdf_edges['raw_danger_score'] / effective_len) * 100

# ─────────────────────────────────────────────
# 4. 3클래스 라벨
# ─────────────────────────────────────────────
print("4. UI 연동용 3클래스 타겟(danger_class) 생성...")
q60, q90 = gdf_edges['final_danger_score'].quantile([0.60, 0.90])
print(f"   -> 분위수 기준점: q60 = {q60:.4f}, q90 = {q90:.4f}")

gdf_edges['danger_class'] = pd.cut(
    gdf_edges['final_danger_score'],
    bins=[-np.inf, q60, q90, np.inf], labels=[0, 1, 2]
).astype(int)

# ─────────────────────────────────────────────
# 5. 저장
# ─────────────────────────────────────────────
print("\n5. 최종 파일 저장...")
if 'name' not in gdf_edges.columns:
    gdf_edges['name'] = 'Unknown'
gdf_edges['name'] = (gdf_edges['name'].astype(str)
                     .replace({'nan': 'Unknown', "['nan']": 'Unknown', 'None': 'Unknown'}))
gdf_edges['osmid'] = gdf_edges['osmid'].astype(str)

final_features = ['u', 'v', 'key', 'osmid', 'name', 'length',
                  'total_crime_score', 'total_request_score',
                  'total_broken_score', 'total_defense_score',
                  'raw_danger_score', 'final_danger_score', 'danger_class', 'geometry']

gdf_final = gdf_edges[final_features].copy()
gdf_final.drop(columns=['geometry']).to_csv("ai_training_features_7km.csv", index=False)
gdf_final.to_file("ai_training_features_7km.geojson", driver='GeoJSON')

end_time = time.time()
print(f"\n🎉 완료! 총 {len(gdf_final):,}개 도로 구간 (소요: {end_time - start_time:.1f}초)")

# ─────────────────────────────────────────────
# 6. 검증
# ─────────────────────────────────────────────
print("\n[검증 1] 클래스 분포 (0:초록, 1:주황, 2:빨강):")
print(gdf_edges['danger_class'].value_counts(normalize=True).sort_index().round(3).to_string())

print("\n[검증 2] final_danger_score 분포:")
print(gdf_edges['final_danger_score'].describe().round(2).to_string())

print("\n[검증 3] 방어점수 반영 확인 (0이면 여전히 문제!):")
print(f"   defense_score < 0 인 도로 수: {(gdf_edges['total_defense_score'] < 0).sum():,}개")

print("\n[검증 4] 가장 위험한 도로 TOP 5:")
print(gdf_final.drop(columns=['geometry'])
      [['name', 'length', 'raw_danger_score', 'final_danger_score', 'danger_class']]
      .sort_values('final_danger_score', ascending=False).head(5).to_string())


🚀 [Step 3] 공간 조인(Spatial Join) 및 가중치 정규화 시작

1. 도로망 데이터 로드 및 정제 중...
   -> 양방향 dedup: 195,320개 → 97,806개
   -> EPSG:26911 변환 및 50m 버퍼 생성 완료

2. 4가지 점 데이터를 도로망에 조인 중...
   ✓ crime_ready.csv: 좌표[정상] | 점 154,032개 → 도로 68,343개 매핑
   ✓ requests_ready.csv: 좌표[정상] | 점 21,329개 → 도로 47,441개 매핑
   ✓ broken_lights_ready.csv: 좌표[정상] | 점 13,727개 → 도로 40,029개 매핑
   ✓ street_lights_ready.csv: 좌표[⚠️ Web Mercator 좌표 감지 → EPSG:4326 자동 변환] | 점 63,051개 → 도로 81,385개 매핑

3. 점수를 양방향 도로망에 매핑 및 밀도 정규화...
4. UI 연동용 3클래스 타겟(danger_class) 생성...
   -> 분위수 기준점: q60 = 79.9057, q90 = 322.0000

5. 최종 파일 저장...

🎉 완료! 총 195,320개 도로 구간 (소요: 11.4초)

[검증 1] 클래스 분포 (0:초록, 1:주황, 2:빨강):
danger_class
0    0.600
1    0.301
2    0.099

[검증 2] final_danger_score 분포:
count    195320.00
mean        120.36
std         238.45
min        -800.00
25%           0.00
50%          46.00
75%         154.00
max        5996.00

[검증 3] 방어점수 반영 확인 (0이면 여전히 문제!):
   defense_score < 0 인 도로 수: 162,544개

[검증 4] 가장 위험한 도로 TOP 5:
           name    

In [73]:
import osmnx as ox
import geopandas as gpd
import time

print("🌐 [Data Prep] ML 패널용 순수 도로망(Edges) 데이터 추출 시작...")
start_time = time.time()

# ML 모델러가 7km 패널 데이터로 작업 중이므로 7000m로 통일합니다.
usc_coords = (34.0224, -118.2851)
radius_meters = 7000

try:
    print(f"⏳ OSM 서버에서 반경 {radius_meters/1000}km 도로망을 다운로드 중입니다. (약 1~3분 소요)")
    # network_type='walk' : 사람이 걸을 수 있는 길만 추출
    G = ox.graph_from_point(usc_coords, dist=radius_meters, network_type='walk')
    
    print("⚙️ 다운로드된 네트워크를 GeoDataFrame으로 변환 및 정제 중...")
    nodes, edges = ox.graph_to_gdfs(G)
    
    # [중요] OSMnx의 edges는 기본적으로 (u, v, key)가 인덱스로 설정되어 있습니다.
    # 패널 데이터 생성기에서 이 값들을 일반 컬럼처럼 써야 하므로 인덱스를 리셋해줍니다.
    edges = edges.reset_index()
    
    # OSMnx 데이터 특성상 일부 속성(예: highway)이 리스트(list)로 들어있는 경우가 있습니다.
    # GeoJSON은 리스트를 지원하지 않아 에러가 나므로, 전부 문자열(str)로 변환합니다.
    for col in edges.columns:
        if any(isinstance(val, list) for val in edges[col]):
            edges[col] = edges[col].astype(str)
            
    output_file = "ml_edges_7km.geojson"
    print(f"💾 데이터를 {output_file} 파일로 저장합니다...")
    edges.to_file(output_file, driver='GeoJSON')
    
    end_time = time.time()
    print(f"\n🎉 성공! 총 {len(edges):,}개의 뼈대 도로 구간(Edges)이 확보되었습니다.")
    print(f"✅ 파일명: {output_file} (소요시간: {end_time - start_time:.2f}초)")
    
    print("\n[필수 컬럼 확인] ML 패널 뼈대용 핵심 컬럼 (u, v, key):")
    print(edges[['u', 'v', 'key', 'length', 'highway']].head())

except Exception as e:
    print(f"\n⚠️ 에러 발생: {e}")
    print("인터넷 연결을 확인하시거나, 잠시 후 다시 시도해 주세요.")

🌐 [Data Prep] ML 패널용 순수 도로망(Edges) 데이터 추출 시작...
⏳ OSM 서버에서 반경 7.0km 도로망을 다운로드 중입니다. (약 1~3분 소요)
⚙️ 다운로드된 네트워크를 GeoDataFrame으로 변환 및 정제 중...
💾 데이터를 ml_edges_7km.geojson 파일로 저장합니다...

🎉 성공! 총 195,320개의 뼈대 도로 구간(Edges)이 확보되었습니다.
✅ 파일명: ml_edges_7km.geojson (소요시간: 21.27초)

[필수 컬럼 확인] ML 패널 뼈대용 핵심 컬럼 (u, v, key):
          u            v  key     length         highway
0  13642118  13103050302    0  89.007840    primary_link
1  13642119   6484564489    0  21.087149  secondary_link
2  14697104   7082081805    0  20.976857         primary
3  14697104  13329054914    0  14.670312         primary
4  14697104  13329054918    0  10.150704       secondary


In [74]:
import pandas as pd
import geopandas as gpd
import numpy as np
import itertools
from datetime import datetime
import time

print("🚀 [Step 5] ML 시계열 패널 데이터(Panel Data) 생성 시작...")
start_time = time.time()

# -------------------------------------------------------------------------
# 1. 기준 시간축(Time Bins) 정의
# -------------------------------------------------------------------------
# period: 2020H1 ~ 2024H2 (반기 단위)
periods = [f"{year}H{half}" for year in range(2020, 2025) for half in [1, 2]]

# dow: 0(월~목), 1(금), 2(토), 3(일)
dows = [0, 1, 2, 3]

# slot: 3시간 단위 8구간 (0: 0~3시, 1: 3~6시 ... 7: 21~24시)
slots = list(range(8))

# 반기별 시작/종료 날짜 매핑 (가로등 활성 상태 계산용)
period_dates = {}
for p in periods:
    year = int(p[:4])
    if p.endswith('H1'):
        period_dates[p] = (pd.to_datetime(f"{year}-01-01"), pd.to_datetime(f"{year}-06-30 23:59:59"))
    else:
        period_dates[p] = (pd.to_datetime(f"{year}-07-01"), pd.to_datetime(f"{year}-12-31 23:59:59"))

# -------------------------------------------------------------------------
# 2. 도로망(Edges) 기반 Base Grid 생성
# -------------------------------------------------------------------------
print("🗺️ 1. 도로망 데이터 로드 및 Base Grid 생성 중...")
gdf_edges = gpd.read_file("ml_edges_7km.geojson")

# ML 모델러 요청: segment_id = (u, v, key)
gdf_edges['segment_id'] = gdf_edges['u'].astype(str) + "_" + gdf_edges['v'].astype(str) + "_" + gdf_edges['key'].astype(str)

# 모든 조합의 뼈대(Grid) 만들기 (Segment x Period x DOW x Slot)
# 주의: 이 작업은 데이터가 수백만 줄이 될 수 있으므로, 실제 환경에서는 Spark 등을 고려해야 할 수도 있습니다.
unique_segments = gdf_edges['segment_id'].unique()
grid = list(itertools.product(unique_segments, periods, dows, slots))
df_panel = pd.DataFrame(grid, columns=['segment_id', 'period', 'dow', 'slot'])
print(f"   -> Base 패널 뼈대 완성: 총 {len(df_panel):,} 행")


# -------------------------------------------------------------------------
# 3. 범죄 데이터(y) 시공간 매핑
# -------------------------------------------------------------------------
print("🚨 2. 범죄 데이터 시공간 매핑 및 Target(y) 생성 중...")
df_crime = pd.read_csv("crime_ready.csv") # feature_engineering.py 결과물
gdf_crime = gpd.GeoDataFrame(df_crime, geometry=gpd.points_from_xy(df_crime['LON'], df_crime['LAT']), crs="EPSG:4326")

# 공간 조인: 범죄가 어느 도로(segment_id)에서 일어났는가? (버퍼 50m 적용)
gdf_edges_proj = gdf_edges.to_crs("EPSG:26911")
gdf_crime_proj = gdf_crime.to_crs("EPSG:26911")
gdf_edges_proj['geometry'] = gdf_edges_proj.geometry.buffer(50)

joined_crime = gpd.sjoin(gdf_crime_proj, gdf_edges_proj[['segment_id', 'geometry']], how='inner', predicate='within')

# 시간 조인: 범죄가 어느 반기, 요일, 시간대인가?
joined_crime['DATE OCC'] = pd.to_datetime(joined_crime['DATE OCC'])
joined_crime['year'] = joined_crime['DATE OCC'].dt.year
joined_crime['month'] = joined_crime['DATE OCC'].dt.month
joined_crime['period'] = joined_crime.apply(lambda row: f"{row['year']}H1" if row['month'] <= 6 else f"{row['year']}H2", axis=1)

# 요일 매핑 (0:월~목, 1:금, 2:토, 3:일)
joined_crime['day_of_week'] = joined_crime['DATE OCC'].dt.dayofweek
joined_crime['dow'] = joined_crime['day_of_week'].map({0:0, 1:0, 2:0, 3:0, 4:1, 5:2, 6:3})

# 시간 매핑 (TIME OCC는 0000 ~ 2359 형태)
joined_crime['slot'] = (joined_crime['TIME OCC'] // 100) // 3
joined_crime['slot'] = joined_crime['slot'].clip(0, 7) # 24시는 7로 클리핑

# 해당 조건에 범죄가 있었는지(1) 없었는지 집계
crime_agg = joined_crime.groupby(['segment_id', 'period', 'dow', 'slot']).size().reset_index(name='crime_count')
crime_agg['y'] = 1 # 범죄가 1건이라도 있으면 타겟 1

# Base 패널에 병합 (범죄가 없었던 곳은 0으로 채움)
df_panel = df_panel.merge(crime_agg[['segment_id', 'period', 'dow', 'slot', 'y']], on=['segment_id', 'period', 'dow', 'slot'], how='left')
df_panel['y'] = df_panel['y'].fillna(0).astype(int)


# -------------------------------------------------------------------------
# 4. 동적 피처: 해당 Period에 활성화된(Active) 고장 가로등 수 계산
# -------------------------------------------------------------------------
print("💡 3. 동적 피처(활성 고장 가로등) 매핑 중...")
df_broken = pd.read_csv("broken_lights_ready.csv")
gdf_broken = gpd.GeoDataFrame(df_broken, geometry=gpd.points_from_xy(df_broken['Longitude'], df_broken['Latitude']), crs="EPSG:4326")
gdf_broken_proj = gdf_broken.to_crs("EPSG:26911")

joined_broken = gpd.sjoin(gdf_broken_proj, gdf_edges_proj[['segment_id', 'geometry']], how='inner', predicate='within')

joined_broken['CreatedDate'] = pd.to_datetime(joined_broken['CreatedDate'], errors='coerce')
joined_broken['ClosedDate'] = pd.to_datetime(joined_broken['ClosedDate'], errors='coerce')

# 각 반기(Period)별로 해당 가로등이 "고장 상태(Active)"였는지 체크
broken_records = []
for p in periods:
    start_date, end_date = period_dates[p]
    
    # 활성 조건: 해당 반기가 끝나기 전에 고장났고 & (아직 안고쳐졌거나 해당 반기 시작 이후에 고쳐짐)
    active_mask = (joined_broken['CreatedDate'] <= end_date) & \
                  (joined_broken['ClosedDate'].isna() | (joined_broken['ClosedDate'] >= start_date))
    
    active_lights = joined_broken[active_mask]
    agg_lights = active_lights.groupby('segment_id').size().reset_index(name='active_broken_lights')
    agg_lights['period'] = p
    broken_records.append(agg_lights)

df_active_broken = pd.concat(broken_records)

# 패널에 병합 (시간/요일 구분 없이 반기별로 동일한 고장 상태를 가짐)
df_panel = df_panel.merge(df_active_broken, on=['segment_id', 'period'], how='left')
df_panel['active_broken_lights'] = df_panel['active_broken_lights'].fillna(0).astype(int)


# -------------------------------------------------------------------------
# 5. [중요] 인접 세그먼트 "과거" 사건율 (Target Leakage 방지)
# -------------------------------------------------------------------------
print("🔗 4. 인접 세그먼트 과거 사건율 (Time-lagged Spatial Feature) 계산 중...")

# 5-1. 네트워크 구조(u, v)를 이용해 이웃 Segment 찾기
edges_df = gdf_edges[['segment_id', 'u', 'v']].copy()
# u나 v를 공유하면 이웃(Adjacent)
neighbors_u = edges_df.merge(edges_df, left_on='u', right_on='u', suffixes=('', '_adj'))
neighbors_v = edges_df.merge(edges_df, left_on='v', right_on='v', suffixes=('', '_adj'))
neighbors_uv = edges_df.merge(edges_df, left_on='u', right_on='v', suffixes=('', '_adj'))
neighbors_vu = edges_df.merge(edges_df, left_on='v', right_on='u', suffixes=('', '_adj'))

all_neighbors = pd.concat([neighbors_u, neighbors_v, neighbors_uv, neighbors_vu])
# 자기 자신 제외 및 중복 제거
all_neighbors = all_neighbors[all_neighbors['segment_id'] != all_neighbors['segment_id_adj']]
all_neighbors = all_neighbors[['segment_id', 'segment_id_adj']].drop_duplicates()

# 5-2. 반기(Period)별 각 segment의 총 범죄 수 사전 계산
period_crime = crime_agg.groupby(['segment_id', 'period'])['crime_count'].sum().reset_index()

# 5-3. 시계열 Shift (이전 반기 데이터 가져오기)
# period 컬럼을 정렬된 카테고리로 변환 후 +1 shift
period_crime['period_idx'] = period_crime['period'].map({p: i for i, p in enumerate(periods)})
period_crime['next_period_idx'] = period_crime['period_idx'] + 1
period_crime['next_period'] = period_crime['next_period_idx'].map({i: p for i, p in enumerate(periods)})

# 다음 반기의 데이터로 매핑 준비 (과거 데이터가 됨)
past_crime = period_crime[['segment_id', 'next_period', 'crime_count']].rename(columns={'next_period': 'period', 'crime_count': 'past_crime_count'})

# 5-4. 이웃 Segment의 과거 범죄 수 합산
adj_past_crime = all_neighbors.merge(past_crime, left_on='segment_id_adj', right_on='segment_id', suffixes=('', '_drop'))
adj_past_crime = adj_past_crime.groupby(['segment_id', 'period'])['past_crime_count'].sum().reset_index()
adj_past_crime.rename(columns={'past_crime_count': 'adj_past_crime_rate'}, inplace=True)

# 패널에 병합
df_panel = df_panel.merge(adj_past_crime, on=['segment_id', 'period'], how='left')
# 첫 반기(2020H1)는 이전 데이터가 없으므로 NaN 그대로 유지 (ML 모델러 요청사항 충족)


# -------------------------------------------------------------------------
# 6. 정적 피처 (길이, 도로 종류 등) 조인 및 저장
# -------------------------------------------------------------------------
print("📊 5. 정적 피처(Static Features) 조인 및 최종 저장 중...")

static_features = gdf_edges[['segment_id', 'length', 'highway', 'oneway']].copy()
df_panel = df_panel.merge(static_features, on='segment_id', how='left')

output_file = "ml_time_series_panel.csv"
df_panel.to_csv(output_file, index=False)

end_time = time.time()
print(f"\n🎉 완벽합니다! 시계열 예측을 위한 최종 패널 데이터가 완성되었습니다.")
print(f"✅ 저장 완료: {output_file} (총 {len(df_panel):,} 행)")
print(f"⏱️ 소요시간: {end_time - start_time:.2f}초")

print("\n[미리보기] 패널 데이터 구조:")
print(df_panel.head(5))

🚀 [Step 5] ML 시계열 패널 데이터(Panel Data) 생성 시작...
🗺️ 1. 도로망 데이터 로드 및 Base Grid 생성 중...
   -> Base 패널 뼈대 완성: 총 62,502,400 행
🚨 2. 범죄 데이터 시공간 매핑 및 Target(y) 생성 중...
💡 3. 동적 피처(활성 고장 가로등) 매핑 중...
🔗 4. 인접 세그먼트 과거 사건율 (Time-lagged Spatial Feature) 계산 중...
📊 5. 정적 피처(Static Features) 조인 및 최종 저장 중...

🎉 완벽합니다! 시계열 예측을 위한 최종 패널 데이터가 완성되었습니다.
✅ 저장 완료: ml_time_series_panel.csv (총 62,502,400 행)
⏱️ 소요시간: 457.33초

[미리보기] 패널 데이터 구조:
               segment_id  period  dow  slot  y  active_broken_lights  \
0  13642118_13103050302_0  2020H1    0     0  0                     0   
1  13642118_13103050302_0  2020H1    0     1  1                     0   
2  13642118_13103050302_0  2020H1    0     2  0                     0   
3  13642118_13103050302_0  2020H1    0     3  0                     0   
4  13642118_13103050302_0  2020H1    0     4  0                     0   

   adj_past_crime_rate    length         highway  oneway  
0                  NaN  89.00784  [primary_link]   False  
1                  NaN  8

In [3]:
import pandas as pd
import geopandas as gpd

print("🗺️ [이슈 1] 세그먼트별 소구역(Subarea) 매핑 테이블 생성 시작...")

# 1. 뼈대 도로망 데이터 불러오기
edges = gpd.read_file("ml_edges_7km.geojson")

# 🚨 [추가된 핵심 코드] u, v, key를 합쳐서 모델러가 쓰는 segment_id를 만들어줍니다!
edges['segment_id'] = edges['u'].astype(str) + "_" + edges['v'].astype(str) + "_" + edges['key'].astype(str)

# 미터법(UTM)으로 좌표계 변환
edges_m = edges.to_crs("EPSG:26911")

# 2. 범죄 데이터에서 'AREA NAME'(소구역 이름)을 활용하기 위해 로드
crime = pd.read_csv("crime_ready.csv")
crime_gpd = gpd.GeoDataFrame(crime, geometry=gpd.points_from_xy(crime['LON'], crime['LAT']), crs="EPSG:4326")
crime_m = crime_gpd.to_crs("EPSG:26911")

# 3. 공간 조인: 가장 가까운 범죄 데이터의 구역 이름을 해당 도로의 소구역으로 간주 (Nearest Join)
joined = gpd.sjoin_nearest(edges_m, crime_m[['AREA NAME', 'geometry']], how='left', distance_col='dist')

# 중복 매핑 제거 (가장 가까운 1개만 남김)
mapping_df = joined.sort_values('dist').drop_duplicates(subset=['segment_id'])

# 모델러가 사용하기 편하게 컬럼 정리 (segment_id, subarea_id)
mapping_final = mapping_df[['segment_id', 'AREA NAME']].rename(columns={'AREA NAME': 'subarea_id'})

# 4. 저장
output_filename = "segment_subarea_mapping.csv"
mapping_final.to_csv(output_filename, index=False)
print(f"✅ 완료! {output_filename}가 생성되었습니다. 모델러는 이 파일을 패널에 조인해서 쓰면 됩니다!")

🗺️ [이슈 1] 세그먼트별 소구역(Subarea) 매핑 테이블 생성 시작...


/Users/jongz/miniconda3/envs/myenv/lib/python3.13/site-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


✅ 완료! segment_subarea_mapping.csv가 생성되었습니다. 모델러는 이 파일을 패널에 조인해서 쓰면 됩니다!


In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import warnings

# 화면에 나타나는 불편한 노란색/빨간색 UserWarning 메시지를 깔끔하게 차단합니다.
warnings.filterwarnings('ignore')

print("🚀 [통합 파이프라인] 데이터 업데이트 작업 시작...\n")

# ==============================================================================
# 1. [이슈 2 해결] 학교/대학 포함 안전 POI 추출 및 GeoJSON 저장 (opening_hours 포함)
# ==============================================================================
print("🏫 [1/2] 학교/대학/교육시설 및 영업시간(opening_hours) 포함 안전 POI 추출 중...")

usc_coords = (34.0224, -118.2851)
radius_meters = 7000

# 모델러 요청 태그 (university, college, school 등 교육 기관 대거 추가)
ml_safe_tags = {
    'amenity': ['police', 'fire_station', 'school', 'university', 'college', 'kindergarten', 'hospital', 'fuel', 'cafe'],
    'shop': ['convenience', 'supermarket'],
    'public_transport': ['station'],
    'railway': ['station']
}

# OSM 데이터 다운로드
pois = ox.features_from_point(usc_coords, tags=ml_safe_tags, dist=radius_meters)

# [경고 차단] Centroid 경고 방지를 위해 UTM 좌표계(EPSG:26911)로 미리 변환 후 점(Centroid) 처리
pois_m = pois.to_crs("EPSG:26911")
pois_m['geometry'] = pois_m.geometry.centroid
pois_clean = pois_m.to_crs("EPSG:4326")

# 필요 컬럼만 정제 ('opening_hours' 필수 추가!)
target_cols = ['name', 'amenity', 'shop', 'public_transport', 'railway', 'opening_hours', 'geometry']
available_cols = [col for col in target_cols if col in pois_clean.columns]
pois_final = pois_clean[available_cols].copy()

poi_output = "ml_safe_pois_7km_v2.geojson"
pois_final.to_file(poi_output, driver="GeoJSON")
print(f"  ✅ [완료] {poi_output} 저장 성공! (총 {len(pois_final):,}개 거점 확보, opening_hours 포함)\n")


# ==============================================================================
# 2. [이슈 1 해결] 세그먼트별 소구역(Subarea) 매핑 테이블 생성
# ==============================================================================
print("🗺️ [2/2] 세그먼트별 소구역(Subarea) 매핑 테이블 생성 중...")

# 2-1. 뼈대 도로망 불러오기
edges = gpd.read_file("ml_edges_7km.geojson")

# u, v, key를 합쳐서 segment_id 생성
edges['segment_id'] = edges['u'].astype(str) + "_" + edges['v'].astype(str) + "_" + edges['key'].astype(str)
edges_m = edges.to_crs("EPSG:26911")

# 2-2. 범죄 데이터 로드 및 좌표 변환
crime = pd.read_csv("crime_ready.csv")
crime_gpd = gpd.GeoDataFrame(crime, geometry=gpd.points_from_xy(crime['LON'], crime['LAT']), crs="EPSG:4326")
crime_m = crime_gpd.to_crs("EPSG:26911")

# 2-3. 공간 Nearest 조인 (가장 가까운 관할 구역 매핑)
joined = gpd.sjoin_nearest(edges_m, crime_m[['AREA NAME', 'geometry']], how='left', distance_col='dist')

# 2-4. 중복 세그먼트 제거 및 최종 컬럼 정제
mapping_df = joined.sort_values('dist').drop_duplicates(subset=['segment_id'])
mapping_final = mapping_df[['segment_id', 'AREA NAME']].rename(columns={'AREA NAME': 'subarea_id'})

mapping_output = "segment_subarea_mapping.csv"
mapping_final.to_csv(mapping_output, index=False)
print(f"  ✅ [완료] {mapping_output} 저장 성공! (총 {len(mapping_final):,}개 세그먼트 매핑 완료)\n")

print("🎉 모든 작업이 에러 없이 깔끔하게 완료되었습니다!")

🚀 [통합 파이프라인] 데이터 업데이트 작업 시작...

🏫 [1/2] 학교/대학/교육시설 포함 안전 POI 추출 중...
  ✅ [완료] ml_safe_pois_7km_v2.geojson 저장 성공! (총 1,130개 거점 확보)

🗺️ [2/2] 세그먼트별 소구역(Subarea) 매핑 테이블 생성 중...
  ✅ [완료] segment_subarea_mapping.csv 저장 성공! (총 195,320개 세그먼트 매핑 완료)

🎉 모든 작업이 에러 없이 깔끔하게 완료되었습니다!


In [2]:
print(edges_m.columns.tolist())

['u', 'v', 'key', 'osmid', 'highway', 'oneway', 'reversed', 'length', 'lanes', 'maxspeed', 'name', 'ref', 'bridge', 'service', 'access', 'width', 'junction', 'tunnel', 'geometry']
